# Ежемесячная подготовка данных — current + dynamic FS

In [ ]:
# Запускается в папке common; импортируем именно общие модули.
from pathlib import Path
import sys, os
p = Path.cwd().resolve()
options = [p, p / 'common', p.parent / 'common']
PROJECT_ROOT = next((x for x in options if (x / 'srcc' / 'nbc_project').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Не найден common/srcc/nbc_project; откройте ноутбук из папки проекта')
os.chdir(PROJECT_ROOT)
SRC_PATH = PROJECT_ROOT / 'srcc'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print('COMMON_PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
Path.cwd()

In [ ]:
# ========== НАСТРОЙКИ ЕЖЕМЕСЯЧНОГО ОБНОВЛЕНИЯ ==========
from nbc_project.feature_lists import load_feature_lists, get_feature_list, select_existing_columns
from datetime import datetime
from dateutil.relativedelta import relativedelta

# В октябрьском продовом запуске добавляли прогнозный месяц сентябрь 2026.
NEW_FORECAST_MONTH = "2026-10-01"
HISTORY_START_MONTH = "2025-09-01"  # старт исторической витрины, НЕ старт окна транзакций
TRAIN_MONTHS = 6                      # история обучения; накопительную витрину НЕ обрезаем
FEATURE_LAG_MONTHS = 7               # глубина истории транзакций для лаговых признаков
POSTFIX_PREFIX = "nbc_monthly"
SAMPLE_SIZE = 250_000
FS_LAG_MONTHS = 2

# Месяц, за который собираем данные
month_dt = datetime.strptime(NEW_FORECAST_MONTH, "%Y-%m-%d")
history_dt = datetime.strptime(HISTORY_START_MONTH, "%Y-%m-%d")
assert month_dt.day == history_dt.day == 1
assert month_dt > history_dt
next_dt = month_dt + relativedelta(months=1)
# Транзакционная история: 6 месяцев назад + текущий месяц
start_date = (
    month_dt - relativedelta(months=6)
).strftime("%Y-%m-%d")
# Правая граница периода (не включительно)
end_date = (
    month_dt + relativedelta(months=1)
).strftime("%Y-%m-%d")
LAST_COMPLETE_MONTH = NEW_FORECAST_MONTH  # совместимость с исходными ячейками
postfix = f"{POSTFIX_PREFIX}_{month_dt:%Y%m}"

print(f"Период транзакций: [{start_date}, {end_date})")

# История: [HISTORY_START_MONTH, NEW_FORECAST_MONTH), новая таблица: + новый месяц.
# Имена повторяют конвенцию из октябрьского продового запуска.
HISTORICAL_FEATURES_TABLE = (
    f"sandbox_crft.mae_nbc_dataset_with_features_{history_dt:%Y%m%d}_{month_dt:%Y%m%d}"
)
DATASET_WITH_FEATURES_UPDATED = (
    f"sandbox_crft.mae_nbc_dataset_with_features_{history_dt:%Y%m%d}_{next_dt:%Y%m%d}"
)

FEATURE_LISTS = load_feature_lists()
print("Добавляем только месяц:", NEW_FORECAST_MONTH)
print("Исторический источник (НЕ перезаписываем):", HISTORICAL_FEATURES_TABLE)
print("Итоговая накопительная таблица:", DATASET_WITH_FEATURES_UPDATED)
print("Окно подготовки транзакций:", start_date, "—", end_date, "(exclusive)")
assert NEW_FORECAST_MONTH == "2026-10-01", "Этот комплект подготовлен для октябрьских таргетов"


In [ ]:
import pandas as pd
import numpy as np
from rail_connectors import connect

# Авторизация на стороне окружения / Kerberos; без паролей в notebook.
conn = connect(to='IMPALA')

In [ ]:
source_tables = [
    'dm_dcp_fl.dim_mcc',
    'dm_dcp_fl.dim_country',
    'dm_dcp_fl.dim_trans_class',
    'dm_dcp_fl.dim_trans_type',
    'dm_dcp_fl.tfct_transaction',
    'ods.scd1_z_r2_kind_cred',
    'acrm.det_customer',
    'acrm.o_categories',
    'acrm.o_merchants',
    'acrm.o_mcc',
    # 'sandbox_crft.dae_loyalty_cat_may_dec',
    'acrm.o_client_categories',
    'acrm.o_client'
]

with conn:
    for table in source_tables:
        conn.execute(f"invalidate metadata {table}")

## Фильтр customer_id, cdi_id

In [ ]:
NBC_CLIENT_IDS = f'sandbox_crft.mae_nbc_client_ids_{postfix}'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {NBC_CLIENT_IDS}''')

In [ ]:
with conn:
    conn.execute(f''' 
        CREATE TABLE {NBC_CLIENT_IDS}
        TBLPROPERTIES('transactional' = 'false')
        AS
        with reff as (
            select
                  customer_id
                , cast(cdi_party_id as bigint) as cdi_id
                , max(
                    case
                        when cust_doc_issue_dt is not null then 1
                        else 0
                    end
                  ) as has_doc_issue_dt
            from dm_dcp_fl.ref_customer
            where cdi_party_id is not null
              and exp_dt > now()
              and source_id = 3
            group by
                  customer_id
                , cast(cdi_party_id as bigint)
        ),
        acrm_clients as (
            select
                cast(party_id as bigint) as cdi_id
            from acrm.o_client
            where party_id is not null
            group by cast(party_id as bigint)
        ),
        marked as (
            select
                  r.customer_id
                , r.cdi_id
                , r.has_doc_issue_dt
                , count(*) over (
                    partition by r.customer_id
                  ) as cnt_cdi_by_customer
                , case
                    when a.cdi_id is not null then 1
                    else 0
                  end as is_in_acrm
                , max(
                    case
                        when a.cdi_id is not null then 1
                        else 0
                    end
                  ) over (
                    partition by r.customer_id
                  ) as customer_has_acrm_cdi
                , max(r.has_doc_issue_dt) over (
                    partition by r.customer_id
                  ) as customer_has_doc_cdi
            from reff r
            left join acrm_clients a
                on r.cdi_id = a.cdi_id
        )
        select
              customer_id
            , cdi_id
        --    , cnt_cdi_by_customer
        --    , is_in_acrm
        --    , has_doc_issue_dt
        --    , customer_has_acrm_cdi
        --    , customer_has_doc_cdi
        from marked
        where
            -- если один cdi_id, ничего не отсекаем
            cnt_cdi_by_customer = 1
            -- если есть cdi_id в acrm.o_client, оставляем только его
            or (
                cnt_cdi_by_customer > 1
                and customer_has_acrm_cdi = 1
                and is_in_acrm = 1
            )
            -- если acrm cdi нет, но есть cdi_id с документом, оставляем его
            or (
                cnt_cdi_by_customer > 1
                and customer_has_acrm_cdi = 0
                and customer_has_doc_cdi = 1
                and has_doc_issue_dt = 1
            )
            -- если ни acrm, ни документа нет, оставляем все cdi_id
            or (
                cnt_cdi_by_customer > 1
                and customer_has_acrm_cdi = 0
                and customer_has_doc_cdi = 0
            );
    ''')

In [ ]:
NBC_CLIENT_IDS

## Данные по транзакциям

In [ ]:
NBC_TRANSACTIONS = f'sandbox_crft.mae_nbc_transactions_{postfix}'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {NBC_TRANSACTIONS}''')

In [ ]:
with conn:
    conn.execute(f''' 
        CREATE TABLE {NBC_TRANSACTIONS}
        TBLPROPERTIES('transactional' = 'false')
        AS (
        select 
        	tt.customer_id, 
        	rc.cdi_id,
        	trans_dsc,
        	tt.mcc_cd,
        	mcc_id,
        	tt.trans_dt,
        	merch_nm,
        	COALESCE(loname.id, 'other') as id,
        	COALESCE(loname.title, 'Другое') as title,
        	abs(tt.charge_rub_amt) as charge_rub_amt
        from 
        (select * from sandbox_crft.pos_trans_1_pl_mae
        	where trans_dt >= '{start_date}'
        	and trans_dt < '{end_date}'
        	) tt
        left join {NBC_CLIENT_IDS} rc
        		on rc.customer_id = tt.customer_id
        left join 
        (select mcc as mcc_cd, category_id
        			from acrm.o_mcc
        			where end_date is null and category_id is not null
        			group by mcc, category_id) lpdmc    --- Мэтчинг МСС и кодов категорий 
                on tt.mcc_cd = lpdmc.mcc_cd  
        left join 
        (select distinct title, category_id
        			from acrm.o_merchants) lmerch    --- Мэтчинг МСС и кодов категорий 
                on tt.trans_dsc = lmerch.title
        left join (select * from acrm.o_categories where end_date is null) loname       --- Мэтчинг кода категории и категории программы лояльности 
            	on loname.id = lpdmc.category_id or loname.id = lmerch.category_id)
''')

In [ ]:
with conn:
    df = conn.fetch(f'''select count(*) from {NBC_TRANSACTIONS} limit 10''')
df

In [ ]:
with conn:
    df = conn.fetch(f'''select * from {NBC_TRANSACTIONS} where trans_dt > '{LAST_COMPLETE_MONTH}' and customer_id is not null limit 10''')
df

## Данные по CDI

In [ ]:
CLIENTS_FOR_LOYALTY = f'sandbox_crft.mae_client_for_loyalty_{postfix}'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {CLIENTS_FOR_LOYALTY}''')

In [ ]:
with conn:
    conn.execute(f'''
    CREATE TABLE {CLIENTS_FOR_LOYALTY}
    TBLPROPERTIES('transactional' = 'false')
    AS with a as (
        select
            customer_id,
            abs(sum(charge_rub_amt))/count(distinct TRUNC(tt.trans_dt, 'mm')) as charge_rub_amt
        from dm_dcp_fl.tfct_transaction tt
        left join dm_dcp_fl.dim_mcc mcc
            on tt.mcc_id = mcc.mcc_id 
        where tt.mcc_id != -99
            -- and charge_rub_amt < 0 
            and customer_id is not null
            and trans_dt >= '{start_date}' and trans_dt < '{end_date}'
        group by customer_id
        ),
        b as (
            select 
                a.customer_id, c.cdi_party_id, count(*)
            from a
            left join dm_dcp_fl.ref_customer c
                on c.customer_id = a.customer_id
            where cdi_party_id is not null
            group by a.customer_id, c.cdi_party_id
        )
        select distinct(cdi_party_id) as cdi_id
        from b
    ''')

In [ ]:
with conn:
    d = conn.fetch(f'''select count(*), count(distinct(cdi_id)) from {CLIENTS_FOR_LOYALTY}''')
    df_all = conn.fetch(f'''select * from {CLIENTS_FOR_LOYALTY} limit 5''')
d

In [ ]:
df_all

## Транзакции по нужным клиентам

In [ ]:
INTEREST_TRANSACTIONS_TABLE = f'sandbox_crft.mae_nbc_transactions_interest_{postfix}'

In [ ]:
def make_months_sql(start_date: str, end_date: str, date_alias: str = "month") -> str:
    """
    Генерирует SQL-блок с месяцами:
    start_date включительно, end_date не включительно.

    Пример:
    start_date='2025-02-01'
    end_date='2026-05-01'

    Вернёт месяцы с 2025-02-01 по 2026-04-01.
    """
    months = pd.date_range(
        start=pd.to_datetime(start_date),
        end=pd.to_datetime(end_date) - pd.DateOffset(months=1),
        freq="MS",
    )

    return "\nUNION ALL\n".join(
        f"select '{month.strftime('%Y-%m-%d')}' as {date_alias}"
        for month in months
    )
months_sql = make_months_sql(start_date, end_date)

In [ ]:
months_sql

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {INTEREST_TRANSACTIONS_TABLE}''')

In [ ]:
with conn:
    conn.execute(f'''
        CREATE TABLE {INTEREST_TRANSACTIONS_TABLE}
        TBLPROPERTIES('transactional' = 'false')
        AS 
        with clients as (
        select 
            cdi_id as cdi_party_id, 
            title as title_, 
            id as id_category, 
            month 
        from (
            select distinct id, title
            from acrm.o_categories 
            where end_date is null
            union all 
            select 'other' as id, 'Другое' as title) cats
        cross join {CLIENTS_FOR_LOYALTY}
        cross join (
                {months_sql}
        	) month
        )
        select * FROM clients c 
        left join {NBC_TRANSACTIONS} t
        on cast(c.cdi_party_id as bigint) = cast(t.cdi_id as bigint)
            and t.id = c.id_category
            and trunc(month, 'MM') = trunc(trans_dt, 'MM')
        ''')

In [ ]:
with conn:
    d = conn.fetch(f'''select cdi_party_id, count(*) from {INTEREST_TRANSACTIONS_TABLE} 
                        group by cdi_party_id
                        limit 10''')
d

In [ ]:
with conn:
    d = conn.fetch(f'''select count(*) from {INTEREST_TRANSACTIONS_TABLE}''')
d

## Уникальные транзакции

In [ ]:
UNIQUE_TRANSACTIONS = f'sandbox_crft.mae_transactions_unique_{postfix}'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {UNIQUE_TRANSACTIONS}''')

In [ ]:
with conn:
    conn.execute(f'''
    CREATE TABLE {UNIQUE_TRANSACTIONS}
    TBLPROPERTIES('transactional' = 'false') as 
    select cdi_party_id as cdi_id, title_ as title, id_category as title_id, month as month_date, trans_dsc, mcc_cd, mcc_id, trans_dt, charge_rub_amt
    from 
    {INTEREST_TRANSACTIONS_TABLE}
    where title_ is not null
    group by cdi_party_id, title_, id_category, month, trans_dsc, mcc_cd, mcc_id, trans_dt, charge_rub_amt''')

In [ ]:
with conn:
    d = conn.fetch(f'''select count(*) from {UNIQUE_TRANSACTIONS}''')
d

## Суммы по категориям

In [ ]:
CHARGES_TABLE = f'sandbox_crft.mae_nbc_categories_charges_{postfix}'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {CHARGES_TABLE}''')

In [ ]:
with conn:
    conn.execute(f'''
    CREATE TABLE {CHARGES_TABLE}
    TBLPROPERTIES('transactional' = 'false')
    AS ( 
    select 
        cdi_id, 
        month_date, 
        title, 
        title_id,
        count(charge_rub_amt) as count_charge, 
        sum(charge_rub_amt) as charge
    from 
    {UNIQUE_TRANSACTIONS}
    where cdi_id is not null
    group by cdi_id, month_date, title, title_id
    --order by cdi_party_id
    )''')


In [ ]:
with conn:
    d = conn.fetch(f'''select count(*) from {CHARGES_TABLE}''')
d

## Join с кешбэком

In [ ]:
CHARGES_TABLE_CASHBACK = f'sandbox_crft.mae_nbc_categories_charges_cashback_{postfix}'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {CHARGES_TABLE_CASHBACK}''')

In [ ]:
with conn:
    conn.execute(f'''
    CREATE TABLE {CHARGES_TABLE_CASHBACK}
    TBLPROPERTIES('transactional' = 'false')
    AS (
        select
            c.*, cast(b.category_percent as double) as cashback, b.source
        from {CHARGES_TABLE} c
        left join sandbox_crft.mae_loyalty_client_categories b
        	on cast(b.party_id as BIGINT) = cast(c.cdi_id as BIGINT)
        	and trunc(b.end_date, 'MM') = trunc(c.month_date, 'MM')
        	and b.category_id = c.title_id
    )''')

In [ ]:
with conn:
    d = conn.fetch(f'''select count(*) from {CHARGES_TABLE_CASHBACK}''')
    d_all = conn.fetch(f'''select * from {CHARGES_TABLE_CASHBACK} where cashback is not null order by cdi_id, month_date limit 10''')
d

In [ ]:
d_all

In [ ]:
with conn:
    d = conn.fetch(f'''select count(*) from {CHARGES_TABLE_CASHBACK}''')
    d_all = conn.fetch(f'''select * from {CHARGES_TABLE_CASHBACK} where cashback is null and cdi_id = 36374601929 order by cdi_id, month_date limit 100''')
d

In [ ]:
d_all

## Предложенные категории

У клиента нужно выбрать те категории, которые он выбирал. Если он выбрал тематические, то по тематическим отсекать

In [ ]:
theme_default = "d40cd3f8-5b8c-426f-84a4-46c848f7b901"

send_to_dbo_base = "dm_dcp_fl.pl_theme_send_to_dbo_v1"
gusev_table = "dm_crft_ml.o_client_theme_clear_cdi"

send_to_dbo_theme = f"sandbox_crft.pl_send_to_dbo_union_theme_{postfix}"

In [ ]:
drop = f"""
drop table if exists {send_to_dbo_theme}
"""

create = f"""
create table {send_to_dbo_theme} stored as parquet as
with gusev_selected as (
    select
    	distinct
        cast(g.cdi_id as bigint) as cdi_id,
        g.theme_id,
        start_date,
        end_date
    from {gusev_table} g
),
client_theme as (
    select
        s.cdi_id,
        coalesce(g.theme_id, 'd40cd3f8-5b8c-426f-84a4-46c848f7b901') as final_theme_id,
        case
            when g.cdi_id is not null then 1
            else 0
        end as selected_from_gusev_flg,
        period_start_date,
        cast(SUBSTRING(start_date, 1, 19) as timestamp),
        cast(SUBSTRING(end_date, 1, 19) as timestamp)
    from (
        select distinct
            cast(cdi_id as bigint) as cdi_id,
            period_start_date
        from {send_to_dbo_base}
    ) s
    left join gusev_selected g
        on s.cdi_id = g.cdi_id
       and period_start_date between cast(SUBSTRING(start_date, 1, 19) as timestamp) and cast(SUBSTRING(end_date, 1, 19) as timestamp)
)
select
    s.id as uuid,
    s.cdi_id,
    s.period_start_date,
    s.category_id,
    s.category_percent
from {send_to_dbo_base} s
join client_theme ct
    on cast(s.cdi_id as bigint) = ct.cdi_id
   and s.theme_id = ct.final_theme_id
   and s.period_start_date = ct.period_start_date
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)

    conn.execute(f"compute stats {send_to_dbo_theme}")

In [ ]:
send_to_dbo = f"sandbox_crft.pl_send_to_dbo_union_{postfix}"

create_send_to_dbo = f"""
    create table {send_to_dbo} as 
    select *
    from dm_dcp_fl.pl_send_to_dbo
    where last_day(cast(period_start_date as  date)) <= '2025-08-31'
    union 
    select *
    from dm_dcp_fl.pl_send_to_dbo_v2
    where last_day(cast(period_start_date as date)) <= '2025-11-30'
    union 
    select *
    from dm_dcp_fl.pl_send_to_dbo_v3
    union 
    select *
    from {send_to_dbo_theme}
    where period_start_date < '{end_date}'
"""

In [ ]:
DATASET = f'sandbox_crft.mae_nbc_dataset_{postfix}'

In [ ]:
with conn:
    for query in [
        f"drop table if exists {DATASET}",
        f"drop table if exists {send_to_dbo}",
        create_send_to_dbo
    ]:
        conn.execute(query)
    df = conn.fetch(f"""select * from {send_to_dbo} where period_start_date = '{LAST_COMPLETE_MONTH}' limit 5""")

In [ ]:
df

## Новые фичи по подключениям

In [ ]:
cb_base = f"sandbox_crft.cb_base_client_month_{postfix}"
cb_charges_compact = f"sandbox_crft.cb_charges_compact_{postfix}"
cb_category_spend_wide = f"sandbox_crft.cb_category_spend_wide_{postfix}"
cb_offered_percent_wide = f"sandbox_crft.cb_offered_percent_wide_{postfix}"
cb_selected_history_wide = f"sandbox_crft.cb_selected_history_wide_{postfix}"
cb_acrm_monthly = f"sandbox_crft.cb_acrm_monthly_{postfix}"
cb_acrm_targets = f"sandbox_crft.cb_acrm_targets_{postfix}"
cb_acrm_lags = f"sandbox_crft.cb_acrm_lags_{postfix}"
cb_activity_features = f"sandbox_crft.cb_activity_features_{postfix}"
cb_potential_features = f"sandbox_crft.cb_potential_features_{postfix}"
cb_share_features = f"sandbox_crft.cb_share_features_{postfix}"
cb_dataset = f"sandbox_crft.cb_client_month_dataset_{postfix}"
cb_total_spend_target = f"sandbox_crft.cb_total_spend_target_{postfix}"

LOYALTY_CLIENT_CATEGORIES = 'sandbox_crft.mae_loyalty_client_categories'
send_to_dbo = f"sandbox_crft.pl_send_to_dbo_union_{postfix}"

def run_sql(conn, queries, compute_stats_table=None, preview_table=None, limit=5):
    with conn:
        for query in queries:
            conn.execute(query)

        if compute_stats_table is not None:
            conn.execute(f"compute stats {compute_stats_table}")

        if preview_table is not None:
            return conn.fetch(f"""
                select *
                from {preview_table}
                limit {limit}
            """)


def category_suffix(category_id: str) -> str:
    return str(category_id).replace("-", "")[-4:]


# start_month/end_month задаются в первой конфигурационной ячейке

#### 1. Базовая таблица клиент-месяц

In [ ]:
drop = f"""
drop table if exists {cb_base}
"""

create = f"""
create table {cb_base} stored as parquet as
select distinct
    cast(cdi_id as bigint) as cdi_id,
    cast(period_start_date as timestamp) as forecast_month
from {send_to_dbo}
where period_start_date >= cast('{start_date}' as timestamp) and period_start_date < cast('{end_date}' as timestamp)
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)
        
    df = conn.fetch(f"""select * from {cb_base} limit 10""")
df

#### 2. Компактная CHARGES TABLE

In [ ]:
cb_charges_compact = f"sandbox_crft.cb_charges_compact_{postfix}"

drop = f"""
drop table if exists {cb_charges_compact}
"""

create = f"""
create table {cb_charges_compact} stored as parquet as
select
    cast(t.cdi_id as bigint) as cdi_id,
    cast(t.month_date as timestamp) as month_date,
    cast(t.title_id as string) as title_id,
    max(t.title) as title,

    sum(coalesce(t.charge, 0)) as charge,
    sum(coalesce(t.count_charge, 0)) as count_charge
from {CHARGES_TABLE} t
join (
    select distinct cdi_id
    from {cb_base}
) b
    on cast(t.cdi_id as bigint) = b.cdi_id
where t.month_date >= add_months(cast('{start_date}' as timestamp), -7)
  and t.month_date < '{end_date}'
  and t.title_id is not null
group by
    cast(t.cdi_id as bigint),
    cast(t.month_date as timestamp),
    cast(t.title_id as string)
"""

run_sql(conn, [drop, create], compute_stats_table=cb_charges_compact, preview_table=cb_charges_compact)

#### 3. Вытаскиваем уникальные категории

In [ ]:
with conn:
    categories_df = conn.fetch(f"""
        select distinct
            title_id,
            title
        from {cb_charges_compact}
        where title_id is not null
        order by title
    """)

categories_df["suffix"] = categories_df["title_id"].apply(category_suffix)

duplicates = (
    categories_df
    .groupby("suffix")
    .size()
    .reset_index(name="cnt")
    .query("cnt > 1")
)

duplicates

#### 4. Общие траты (+ target) + wide-траты в категориях

In [ ]:
def generate_category_spend_sql(categories_df):
    sql_parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        sql_parts.append(f"""
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -2)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag1m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -3)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag2m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -4)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -4)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_6m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -4)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.count_charge, 0) else 0
            end) as cnt_cat_{suffix}_3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.count_charge, 0) else 0
            end) as cnt_cat_{suffix}_6m
        """)

    return ",\n".join(sql_parts)

In [ ]:
category_spend_sql = generate_category_spend_sql(categories_df)

drop = f"""
drop table if exists {cb_category_spend_wide}
"""

create = f"""
create table {cb_category_spend_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    -- общие траты клиента, lag_1m = последний закрытый месяц
    sum(case
            when t.month_date = add_months(b.forecast_month, -2)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_1m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -3)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_2m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -4)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_3m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -5)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_4m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -6)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_5m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -7)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_6m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -4)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_3m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_6m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -2)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_lag_1m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -4)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_3m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_6m,
    -- wide-фичи по категориям
    {category_spend_sql}
from {cb_base} b
left join {cb_charges_compact} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_category_spend_wide, preview_table=cb_category_spend_wide)

In [ ]:
# Таргет общего POS-оборота
drop = f"""
    drop table if exists {cb_total_spend_target}
"""

create = f"""
    create table {cb_total_spend_target} stored as parquet as
    select
        b.cdi_id,
        b.forecast_month,
        sum(coalesce(t.charge, 0)) as target_total_spend_month,
        sum(coalesce(t.count_charge, 0)) as target_total_cnt_txn_month
    from {cb_base} b
    left join {cb_charges_compact} t
        on b.cdi_id = t.cdi_id
       and t.month_date = b.forecast_month
    group by
        b.cdi_id,
        b.forecast_month
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)

    conn.execute(f"compute stats {cb_total_spend_target}")

    df_check = conn.fetch(f"""
    select *
    from {cb_total_spend_target}
    limit 10
    """)

df_check

#### 5. Предложенные категории

In [ ]:
def generate_offered_percent_sql(categories_df):
    sql_parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        sql_parts.append(f"""
        max(case
                when o.category_id = '{category_id}'
                then cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)
                else 0
            end) as offered_percent_cat_{suffix}
        """)

    return ",\n".join(sql_parts)

offered_percent_sql = generate_offered_percent_sql(categories_df)

drop = f"""
drop table if exists {cb_offered_percent_wide}
"""

create = f"""
create table {cb_offered_percent_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,
    count(distinct o.category_id) as num_offered_categories,
    max(cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)) as max_offered_percent,
    avg(cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)) as avg_offered_percent,
    {offered_percent_sql}
from {cb_base} b
left join {send_to_dbo} o
    on b.cdi_id = cast(o.cdi_id as bigint)
   and b.forecast_month = cast(o.period_start_date as timestamp)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_offered_percent_wide, preview_table=cb_offered_percent_wide)

In [ ]:
cb_offered_percent_wide

#### 6. История подключенных категорий

In [ ]:
def generate_selected_history_sql(categories_df):
    parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        parts.append(f"""
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
                then 1 else 0
            end) as was_selected_cat_{suffix}_lag1m,
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then 1 else 0
            end) as was_selected_cat_{suffix}_3m,
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then 1 else 0
            end) as was_selected_cat_{suffix}_6m,
        avg(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
            end) as avg_selected_percent_cat_{suffix}_6m
        """)

    return ",\n".join(parts)

In [ ]:
selected_history_sql = generate_selected_history_sql(categories_df)

drop = f"""
drop table if exists {cb_selected_history_wide}
"""

create = f"""
create table {cb_selected_history_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    case
        when count(distinct case
            when trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
            then s.category_id
        end) > 0
        then 1 else 0
    end as selected_any_lag_1m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
        then s.category_id
    end) as cnt_selected_categories_lag_1m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then s.category_id
    end) as cnt_unique_selected_categories_3m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then s.category_id
    end) as cnt_unique_selected_categories_6m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then concat(cast(trunc(cast(s.end_date as timestamp), 'MM') as string), '', cast(s.category_id as string))
    end) as cnt_selected_category_months_3m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then concat(cast(trunc(cast(s.end_date as timestamp), 'MM') as string), '', cast(s.category_id as string))
    end) as cnt_selected_category_months_6m,
    max(case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
    end) as max_selected_percent_6m,
    avg(case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
    end) as avg_selected_percent_6m,
    {selected_history_sql}
from {cb_base} b
left join {LOYALTY_CLIENT_CATEGORIES} s
    on b.cdi_id = cast(s.party_id as bigint)
   and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
   and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(
    conn,
    [drop, create],
    compute_stats_table=cb_selected_history_wide,
    preview_table=cb_selected_history_wide
)

#### 7. ACRM targets

In [ ]:
drop = f"""
drop table if exists {cb_acrm_monthly}
"""

create = f"""
create table {cb_acrm_monthly} stored as parquet as
select
    cast(b.party_id as bigint) as cdi_id,
    cast(trunc(cast(a.transaction_date as date), 'MM') as timestamp) as month_date,
    least(sum(coalesce(a.coins, 0)), 5000) as cashback_month,
    sum(coalesce(a.transaction_amount, 0)) as pos_turnover_month,
    count(*) as pos_txn_cnt_month
from acrm.o_calculations a
join acrm.o_client b
    on a.client_id = b.id
join (
    select distinct cdi_id
    from {cb_base}
) base
    on cast(b.party_id as bigint) = base.cdi_id
where cast(a.transaction_date as date) >= add_months(cast('{start_date}' as timestamp), -7)
  and cast(a.transaction_date as date) < add_months(cast('{end_date}' as timestamp), 1)
group by
    cast(b.party_id as bigint),
    cast(trunc(cast(a.transaction_date as date), 'MM') as timestamp)
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_monthly, preview_table=cb_acrm_monthly)

In [ ]:
drop = f"""
drop table if exists {cb_acrm_targets}
"""

create = f"""
create table {cb_acrm_targets} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,
    coalesce(t.cashback_month, 0) as target_cashback_month,
    coalesce(t.pos_turnover_month, 0) as target_pos_turnover_month,
    coalesce(t.pos_txn_cnt_month, 0) as target_pos_txn_cnt_month
from {cb_base} b
left join {cb_acrm_monthly} t
    on b.cdi_id = t.cdi_id
   and b.forecast_month = t.month_date
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_targets, preview_table=cb_acrm_targets)

#### 8. ACRM lags

In [ ]:
drop = f"""
drop table if exists {cb_acrm_lags}
"""

create = f"""
create table {cb_acrm_lags} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    sum(case when t.month_date = add_months(b.forecast_month, -2)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_1m,
    sum(case when t.month_date = add_months(b.forecast_month, -3)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_2m,
    sum(case when t.month_date = add_months(b.forecast_month, -4)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_6m,
    avg(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) end) as cashback_avg_3m,
    stddev(case when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.cashback_month, 0) end) as cashback_std_6m,
    sum(case when t.month_date = add_months(b.forecast_month, -2)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_1m,
    sum(case when t.month_date = add_months(b.forecast_month, -3)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_2m,
    sum(case when t.month_date = add_months(b.forecast_month, -4)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_6m,
    avg(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) end) as pos_turnover_avg_3m,
    stddev(case when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.pos_turnover_month, 0) end) as pos_turnover_std_6m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_txn_cnt_month, 0) else 0 end) as pos_txn_cnt_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_txn_cnt_month, 0) else 0 end) as pos_txn_cnt_6m
from {cb_base} b
left join {cb_acrm_monthly} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_lags, preview_table=cb_acrm_lags)

#### 9. Activity feats

In [ ]:
drop = f"""
drop table if exists {cb_activity_features}
"""

create = f"""
create table {cb_activity_features} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    count(distinct case
        when t.month_date = add_months(b.forecast_month, -2)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_1m,
    count(distinct case
        when t.month_date >= add_months(b.forecast_month, -4)
         and t.month_date < add_months(b.forecast_month, -1)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_3m,
    count(distinct case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_6m,
    stddev(case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
        then coalesce(t.charge, 0)
    end) as category_spend_std_6m,
    avg(case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
        then coalesce(t.charge, 0)
    end) as category_spend_avg_6m,
    case
        when avg(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0)
        end) > 0
        then
            stddev(case
                when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0)
            end)
            /
            avg(case
                when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0)
            end)
        else 0
    end as category_spend_cv_6m
from {cb_base} b
left join {cb_charges_compact} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_activity_features, preview_table=cb_activity_features)

#### 10. Потенциальный кешбэк

In [ ]:
drop = f"""
drop table if exists {cb_potential_features}
"""

create = f"""
create table {cb_potential_features} stored as parquet as
with spend_3m_by_category as (
    select
        b.cdi_id,
        b.forecast_month,
        t.title_id as category_id,
        sum(coalesce(t.charge, 0)) as spend_3m,
        sum(coalesce(t.count_charge, 0)) as cnt_txn_3m
    from {cb_base} b
    join {cb_charges_compact} t
        on b.cdi_id = t.cdi_id
       and t.month_date >= add_months(b.forecast_month, -4)
       and t.month_date < add_months(b.forecast_month, -1)
    group by
        b.cdi_id,
        b.forecast_month,
        t.title_id
),
offered as (
    select
        cast(cdi_id as bigint) as cdi_id,
        cast(period_start_date as timestamp) as forecast_month,
        category_id,
        max(cast(regexp_replace(cast(category_percent as string), ',', '.') as double)) as offered_percent
    from {send_to_dbo}
    group by
        cast(cdi_id as bigint),
        cast(period_start_date as timestamp),
        category_id
)
select
    b.cdi_id,
    b.forecast_month,

    sum(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_sum_offered,
    max(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_max_offered,
    avg(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_avg_offered,
    count(distinct case
        when coalesce(s.spend_3m, 0) > 0
        then o.category_id
    end) as cnt_offered_categories_with_spend_3m,
    sum(coalesce(s.spend_3m, 0)) as offered_categories_spend_3m,
    sum(coalesce(s.cnt_txn_3m, 0)) as offered_categories_cnt_txn_3m
from {cb_base} b
left join offered o
    on b.cdi_id = o.cdi_id
   and b.forecast_month = o.forecast_month
left join spend_3m_by_category s
    on o.cdi_id = s.cdi_id
   and o.forecast_month = s.forecast_month
   and o.category_id = s.category_id
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_potential_features, preview_table=cb_potential_features)

#### 11. Share-фичи

In [ ]:
categories_df

In [ ]:
def generate_share_sql(categories_df):
    parts = []

    for _, row in categories_df.iterrows():
        suffix = row["suffix"]

        parts.append(f"""
        case
            when coalesce(total_spend_3m, 0) > 0
            then coalesce(spend_cat_{suffix}_3m, 0) / total_spend_3m
            else 0
        end as share_spend_cat_{suffix}_3m
        """)

    return ",\n".join(parts)


share_sql = generate_share_sql(categories_df)

In [ ]:
drop = f"""
drop table if exists {cb_share_features}
"""

create = f"""
create table {cb_share_features} stored as parquet as
select
    cdi_id,
    forecast_month,
    {share_sql}
from {cb_category_spend_wide}
"""

run_sql(conn, [drop, create], compute_stats_table=cb_share_features, preview_table=cb_share_features)

In [ ]:
share_cols_sql = ",\n    ".join([
    f"coalesce(sh.share_spend_cat_{row['suffix']}_3m, 0) as sharespend_cat_{row['suffix']}_3m"
    for _, row in categories_df.iterrows()
])

selected_history_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]

    selected_history_cols.extend([
        f"coalesce(sel.was_selected_cat_{suffix}_lag1m, 0) as was_selected_cat_{suffix}_lag1m",
        f"coalesce(sel.was_selected_cat_{suffix}_3m, 0) as wasselected_cat_{suffix}_3m",
        f"coalesce(sel.was_selected_cat_{suffix}_6m, 0) as wasselected_cat_{suffix}_6m",
        f"coalesce(sel.avg_selected_percent_cat_{suffix}_6m, 0) as avgselected_percent_cat_{suffix}_6m",
    ])

selected_history_cols_sql = ",\n    ".join(selected_history_cols)

In [ ]:
spend_wide_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]
    spend_wide_cols.extend([
        f"coalesce(sp.spend_cat_{suffix}_lag1m, 0) as spend_cat_{suffix}_lag1m",
        f"coalesce(sp.spend_cat_{suffix}_lag2m, 0) as spend_cat_{suffix}_lag2m",
        f"coalesce(sp.spend_cat_{suffix}_lag3m, 0) as spend_cat_{suffix}_lag3m",
        f"coalesce(sp.spend_cat_{suffix}_3m, 0) as spendcat_{suffix}_3m",
        f"coalesce(sp.spend_cat_{suffix}_6m, 0) as spendcat_{suffix}_6m",
        f"coalesce(sp.cnt_cat_{suffix}_3m, 0) as cntcat_{suffix}_3m",
        f"coalesce(sp.cnt_cat_{suffix}_6m, 0) as cntcat_{suffix}_6m",
    ])

spend_wide_cols_sql = ",\n    ".join(spend_wide_cols)


offered_wide_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]
    offered_wide_cols.append(
        f"coalesce(off.offered_percent_cat_{suffix}, 0) as offered_percent_cat_{suffix}"
    )

offered_wide_cols_sql = ",\n    ".join(offered_wide_cols)

#### 12. Финальная витрина

In [ ]:
drop = f"""
drop table if exists {cb_dataset}
"""

create = f"""
create table {cb_dataset} stored as parquet as
with loyalty_connected as (
    select
        cast(party_id as bigint) as cdi_id,
        trunc(cast(end_date as timestamp), 'MM') as forecast_month
    from {LOYALTY_CLIENT_CATEGORIES}
    group by
        cast(party_id as bigint),
        trunc(cast(end_date as timestamp), 'MM')
)
select
    b.cdi_id,
    b.forecast_month,
    -- targets
    coalesce(trg.target_cashback_month, 0) as target_cashback_month,
    coalesce(trg.target_pos_turnover_month, 0) as target_pos_turnover_month,
    coalesce(trg.target_pos_txn_cnt_month, 0) as target_pos_txn_cnt_month,
    coalesce(tst.target_total_spend_month, 0) as target_total_spend_month,
    case
        when lc.cdi_id is not null then 1
        else 0
    end as target_loyalty_connected_flg,
    -- общие траты клиента
    coalesce(sp.total_spend_lag_1m, 0) as total_spend_lag_1m,
    coalesce(sp.total_spend_lag_2m, 0) as total_spend_lag_2m,
    coalesce(sp.total_spend_lag_3m, 0) as total_spend_lag_3m,
    coalesce(sp.total_spend_lag_4m, 0) as total_spend_lag_4m,
    coalesce(sp.total_spend_lag_5m, 0) as total_spend_lag_5m,
    coalesce(sp.total_spend_lag_6m, 0) as total_spend_lag_6m,
    coalesce(sp.total_spend_3m, 0) as total_spend_3m,
    coalesce(sp.total_spend_6m, 0) as total_spend_6m,
    coalesce(sp.total_cnt_txn_lag_1m, 0) as total_cnt_txn_lag_1m,
    coalesce(sp.total_cnt_txn_3m, 0) as total_cnt_txn_3m,
    coalesce(sp.total_cnt_txn_6m, 0) as total_cnt_txn_6m,
    -- wide-траты по категориям
    {spend_wide_cols_sql},
    -- предложенные категории на forecast_month
    coalesce(off.num_offered_categories, 0) as num_offered_categories,
    coalesce(off.max_offered_percent, 0) as max_offered_percent,
    coalesce(off.avg_offered_percent, 0) as avg_offered_percent,
    {offered_wide_cols_sql},
    -- история выбранных категорий
    coalesce(sel.selected_any_lag_1m, 0) as selected_any_lag_1m,
    coalesce(sel.cnt_selected_categories_lag_1m, 0) as cnt_selected_categories_lag_1m,
    coalesce(sel.cnt_unique_selected_categories_3m, 0) as cnt_unique_selected_categories_3m,
    coalesce(sel.cnt_unique_selected_categories_6m, 0) as cnt_unique_selected_categories_6m,
    coalesce(sel.max_selected_percent_6m, 0) as max_selected_percent_6m,
    coalesce(sel.avg_selected_percent_6m, 0) as avg_selected_percent_6m,
    {selected_history_cols_sql},
    -- ACRM lags
    coalesce(al.cashback_lag_1m, 0) as cashback_lag_1m,
    coalesce(al.cashback_lag_2m, 0) as cashback_lag_2m,
    coalesce(al.cashback_lag_3m, 0) as cashback_lag_3m,
    coalesce(al.cashback_3m, 0) as cashback_3m,
    coalesce(al.cashback_6m, 0) as cashback_6m,
    coalesce(al.cashback_avg_3m, 0) as cashback_avg_3m,
    coalesce(al.cashback_std_6m, 0) as cashback_std_6m,
    coalesce(al.pos_turnover_lag_1m, 0) as pos_turnover_lag_1m,
    coalesce(al.pos_turnover_lag_2m, 0) as pos_turnover_lag_2m,
    coalesce(al.pos_turnover_lag_3m, 0) as pos_turnover_lag_3m,
    coalesce(al.pos_turnover_3m, 0) as pos_turnover_3m,
    coalesce(al.pos_turnover_6m, 0) as pos_turnover_6m,
    coalesce(al.pos_turnover_avg_3m, 0) as pos_turnover_avg_3m,
    coalesce(al.pos_turnover_std_6m, 0) as pos_turnover_std_6m,
    coalesce(al.pos_txn_cnt_3m, 0) as pos_txn_cnt_3m,
    coalesce(al.pos_txn_cnt_6m, 0) as pos_txn_cnt_6m,
    -- активность
    coalesce(act.cnt_categories_with_spend_1m, 0) as cnt_categories_with_spend_1m,
    coalesce(act.cnt_categories_with_spend_3m, 0) as cnt_categories_with_spend_3m,
    coalesce(act.cnt_categories_with_spend_6m, 0) as cnt_categories_with_spend_6m,
    coalesce(act.category_spend_std_6m, 0) as category_spend_std_6m,
    coalesce(act.category_spend_avg_6m, 0) as category_spend_avg_6m,
    coalesce(act.category_spend_cv_6m, 0) as category_spend_cv_6m,
    -- potential cashback
    coalesce(p.potential_cashback_sum_offered, 0) as potential_cashback_sum_offered,
    coalesce(p.potential_cashback_max_offered, 0) as potential_cashback_max_offered,
    coalesce(p.potential_cashback_avg_offered, 0) as potential_cashback_avg_offered,
    coalesce(p.cnt_offered_categories_with_spend_3m, 0) as cnt_offered_categories_with_spend_3m,
    coalesce(p.offered_categories_spend_3m, 0) as offered_categories_spend_3m,
    coalesce(p.offered_categories_cnt_txn_3m, 0) as offered_categories_cnt_txn_3m,
    -- share categories
    {share_cols_sql},
    -- calendar
    year(b.forecast_month) as forecast_year,
    month(b.forecast_month) as forecast_month_num
from {cb_base} b
left join {cb_acrm_targets} trg
    on b.cdi_id = trg.cdi_id
   and b.forecast_month = trg.forecast_month
left join {cb_category_spend_wide} sp
    on b.cdi_id = sp.cdi_id
   and b.forecast_month = sp.forecast_month
left join {cb_offered_percent_wide} off
    on b.cdi_id = off.cdi_id
   and b.forecast_month = off.forecast_month
left join {cb_selected_history_wide} sel
    on b.cdi_id = sel.cdi_id
   and b.forecast_month = sel.forecast_month
left join {cb_acrm_lags} al
    on b.cdi_id = al.cdi_id
   and b.forecast_month = al.forecast_month
left join {cb_activity_features} act
    on b.cdi_id = act.cdi_id
   and b.forecast_month = act.forecast_month
left join {cb_potential_features} p
    on b.cdi_id = p.cdi_id
   and b.forecast_month = p.forecast_month
left join {cb_share_features} sh
    on b.cdi_id = sh.cdi_id
   and b.forecast_month = sh.forecast_month
left join {cb_total_spend_target} tst
    on b.cdi_id = tst.cdi_id
   and b.forecast_month = tst.forecast_month
left join loyalty_connected lc
    on b.cdi_id = lc.cdi_id
   and b.forecast_month = lc.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_dataset, preview_table=cb_dataset)

In [ ]:
cb_dataset

## Фичестор

In [ ]:
with conn:
    d1 = conn.fetch(f'''select * from {cb_dataset} where forecast_month = '{LAST_COMPLETE_MONTH}' limit 10''')
    df_all = conn.fetch(f'''select count(*) from {cb_dataset} limit 5''')
d1

In [ ]:
df_all

In [ ]:
with conn:
    d3 = conn.fetch(f'''select * from acrm.common_segmentation limit 5''')
d3

In [ ]:
data_cols = list(d1.columns)
cs_cols = [col for col in d3.columns if col not in data_cols and col != 'cdi_id']

data_cols

In [ ]:
# Не допускаем одинаковых имён столбцов из базы и common_segmentation.
assert len(data_cols) == len(set(data_cols)), 'Дубли в cb_dataset'


In [ ]:
set(cs_cols) & set(data_cols)

In [ ]:
set(cs_cols) | set(data_cols)

In [ ]:
# Все базовые поля квалифицируем явно, чтобы избежать неоднозначных имён.
data_cols_sql = [f"ld.`{col}`" for col in data_cols]
cs_cols_sql = [f"cs.`{col}`" for col in cs_cols]


In [ ]:
data_cols

In [ ]:
select_clause = ", ".join(data_cols_sql + cs_cols_sql)


In [ ]:
DATASET_WITH_FEATURES = f'sandbox_crft.mae_nbc_dataset_with_features_{postfix}'
DATASET_WITH_FEATURES

In [ ]:
# Для CTAS в той же Impala, что и остальные этапы.
# Если нужен отдельный сервисный коннектор — настройте его через окружение, не в notebook.
conn_hive = conn


In [ ]:
with conn:
    conn.execute(f'''drop table if exists {DATASET_WITH_FEATURES}''')

### common_segmetation

In [ ]:
def make_month_list(start_date: str, end_date: str) -> list[str]:
    """
    Генерирует список месяцев:
    start_date + 1 месяц включительно,
    end_date не включительно.

    Например:
    start_date = '2025-09-01'
    end_date   = '2026-06-01'

    Результат:
    ['2025-10-01', ..., '2026-05-01']
    """
    return [
        d.strftime("%Y-%m-%d")
        for d in pd.date_range(
            start=pd.to_datetime(start_date) + pd.DateOffset(months=1),
            end=pd.to_datetime(end_date) - pd.DateOffset(months=1),
            freq="MS",
        )
    ]

In [ ]:
# В каждый запуск собираем только новый прогнозный месяц.
months = [NEW_FORECAST_MONTH]
print("Новый прогнозный месяц:", months)

In [ ]:
# CURRENT: временная таблица ТОЛЬКО за новый месяц. Исторические данные не трогаем.
with conn:
    conn.execute(f"""
        CREATE TABLE {DATASET_WITH_FEATURES}
        STORED AS PARQUET TBLPROPERTIES('transactional'='false') AS
        SELECT {select_clause}
        FROM {cb_dataset} ld
        LEFT JOIN acrm.common_segmentation cs
          ON cast(cs.cdi_id as bigint) = cast(ld.cdi_id as bigint)
         AND trunc(cast(cs.report_date as date), 'MM') =
             add_months(trunc(cast(ld.forecast_month as date), 'MM'), -1)
        WHERE cast(ld.forecast_month as date) = '{NEW_FORECAST_MONTH}'
    """)

with conn:
    new_month_check = conn.fetch(f"""
        SELECT cast(forecast_month as date) AS forecast_month, COUNT(*) AS rows_cnt
        FROM {DATASET_WITH_FEATURES} GROUP BY cast(forecast_month as date)
    """)
assert len(new_month_check) == 1 and str(pd.to_datetime(new_month_check.iloc[0]['forecast_month']).date()) == NEW_FORECAST_MONTH, (
    'Новый срез должен содержать ровно один месяц: ' + NEW_FORECAST_MONTH
)
print(new_month_check)

In [ ]:
DATASET_WITH_FEATURES

In [ ]:
# Восстанавливаем продовую схему: история + новый срез; при повторном расчёте месяца заменяем его.
# В отличие от DROP + CREATE, не удаляем исходную накопительную таблицу.
assert HISTORICAL_FEATURES_TABLE != DATASET_WITH_FEATURES_UPDATED
assert DATASET_WITH_FEATURES_UPDATED != DATASET_WITH_FEATURES
with conn:
    conn.execute(f"""
        CREATE TABLE {DATASET_WITH_FEATURES_UPDATED}
        STORED AS PARQUET
        TBLPROPERTIES('transactional' = 'false') AS
        SELECT *
        FROM {HISTORICAL_FEATURES_TABLE}
        WHERE cast(forecast_month as date) <> '{NEW_FORECAST_MONTH}'
        UNION ALL
        SELECT * FROM {DATASET_WITH_FEATURES}
    """)

    updated_months_check = conn.fetch(f"""
        SELECT cast(forecast_month as date) AS forecast_month, COUNT(*) AS rows_cnt
        FROM {DATASET_WITH_FEATURES_UPDATED}
        GROUP BY cast(forecast_month as date)
        ORDER BY forecast_month
    """)

assert NEW_FORECAST_MONTH in set(pd.to_datetime(updated_months_check['forecast_month']).dt.strftime('%Y-%m-%d'))
assert updated_months_check['rows_cnt'].gt(0).all()
print('Обновлённая накопительная таблица:', DATASET_WITH_FEATURES_UPDATED)
print(updated_months_check)

In [ ]:
# Все последующие шаги должны использовать накопительную таблицу (история + новый месяц),
# а не временный одно-месячный срез.
DATASET_WITH_FEATURES_NEW_MONTH = DATASET_WITH_FEATURES
DATASET_WITH_FEATURES = DATASET_WITH_FEATURES_UPDATED
print('CURRENT / источник для обучения:', DATASET_WITH_FEATURES)

In [ ]:
# Контроль: прежняя историческая таблица не была изменена.
# Исключённый новый месяц не удваивается из-за UNION ALL.
with conn:
    historical_counts = conn.fetch(f"""
        SELECT cast(forecast_month as date) AS m, count(*) AS n
        FROM {HISTORICAL_FEATURES_TABLE}
        WHERE cast(forecast_month as date) <> '{NEW_FORECAST_MONTH}'
        GROUP BY cast(forecast_month as date)
    """)
    updated_counts = conn.fetch(f"""
        SELECT cast(forecast_month as date) AS m, count(*) AS n
        FROM {DATASET_WITH_FEATURES_UPDATED}
        WHERE cast(forecast_month as date) <> '{NEW_FORECAST_MONTH}'
        GROUP BY cast(forecast_month as date)
    """)
assert historical_counts.sort_values('m').reset_index(drop=True).equals(
    updated_counts.sort_values('m').reset_index(drop=True)
), 'Изменились исторические количества строк'
print('Исторические месяцы сохранены без изменений по количеству строк')

In [ ]:
# Никакого DROP для HISTORICAL_FEATURES_TABLE: она остаётся точкой отката.
print('История сохранена:', HISTORICAL_FEATURES_TABLE)

### feature store modular

In [ ]:
with conn:
    d2 = conn.fetch(f'''select * from sandbox_crft.feature_store_inc_modular limit 5''')
d2

In [ ]:
fs_cols = [
    col for col in d2.columns
    if col not in data_cols and col not in {'hash_num','cdi_id','report_date'}
    and not (col.startswith('loyalty__') or col.endswith(('_date','_dttm','_cust_id')))
]


In [ ]:
assert len(fs_cols) == len(set(fs_cols)), 'Дубли имён фичестора'


In [ ]:
# Берём из обновлённого CURRENT только исходные поля/таргеты cb_dataset,
# не переносим признаки common_segmentation в динамический датасет.
# Дополнительные признаки загружаем исключительно из modular FS.
select_clause_fs = ", ".join([f"ld.`{col}`" for col in data_cols] + [f"cs.`{col}`" for col in fs_cols])
assert not set(fs_cols) & set(data_cols)


In [ ]:
# Список колонок для dynamic-среза построен в предыдущей ячейке.


In [ ]:
DATASET_WITH_FEATURES_FS = f'sandbox_crft.mae_nbc_dataset_with_features_fs_{postfix}'

In [ ]:
DATASET_WITH_FEATURES_FS

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {DATASET_WITH_FEATURES_FS}''')

In [ ]:
print(f'Динамических признаков: {len(fs_cols)}')


In [ ]:
# DYNAMIC строится ИЗ НАКОПИТЕЛЬНОЙ CURRENT после добавления нового месяца.
# Из current берём исходные поля cb_dataset (включая таргеты), но не common_segmentation.
# FS_LAG_MONTHS=2 сохранён из исходного эксперимента.
with conn_hive:
    conn_hive.execute(f"""
        CREATE TABLE {DATASET_WITH_FEATURES_FS}
        STORED AS PARQUET TBLPROPERTIES('transactional'='false') AS
        SELECT {select_clause_fs}
        FROM {DATASET_WITH_FEATURES_UPDATED} ld
        LEFT JOIN sandbox_crft.feature_store_inc_modular cs
          ON cast(cs.cdi_id as bigint) = cast(ld.cdi_id as bigint)
         AND trunc(cast(cs.report_date as date), 'MM') =
             add_months(trunc(cast(ld.forecast_month as date), 'MM'), -{FS_LAG_MONTHS})
    """)


## Контроль качества двух источников перед обучением

In [ ]:
# Обязательные проверки: все месяцы и количество строк совпадают между подходами.
with conn:
    compare_months = conn.fetch(f"""
        SELECT 'current' AS approach, cast(forecast_month as date) AS m,
               count(*) AS rows_cnt, count(distinct cdi_id) AS clients_cnt
        FROM {DATASET_WITH_FEATURES} GROUP BY cast(forecast_month as date)
        UNION ALL
        SELECT 'dynamic' AS approach, cast(forecast_month as date) AS m,
               count(*) AS rows_cnt, count(distinct cdi_id) AS clients_cnt
        FROM {DATASET_WITH_FEATURES_FS} GROUP BY cast(forecast_month as date)
        ORDER BY m, approach
    """)
print(compare_months)
assert compare_months.groupby('approach')['m'].nunique().eq(FORECAST_MONTHS).all(), 'Не все месяцы собраны'
counts = compare_months.pivot(index='m', columns='approach', values='rows_cnt')
assert (counts['current'] == counts['dynamic']).all(), 'Не совпадает число строк current/dynamic'


## Витрина для обучения подключения лояльности

In [ ]:
# CURRENT/static feature list — оставляем как в текущем pipeline.
cols_bin = get_feature_list('train_pipeline_loyalty_binary_columns')

cols_bin_sql = ",".join(cols_bin)

In [ ]:
%%time
DATASET_BINARY_LOYAL_SAMPLE = f"sandbox_crft.cb_sample_for_binary_{postfix}"
# SAMPLE_SIZE берём из конфигурации в начале ноутбука

query = f"""
    CREATE TABLE {DATASET_BINARY_LOYAL_SAMPLE}
    stored as parquet as
    --PARTITIONED BY (period)
    WITH sample_keys AS (
        SELECT cdi_id, forecast_month
        FROM (
            SELECT
                t.cdi_id,
                t.forecast_month,
                ROW_NUMBER() OVER (
                    PARTITION BY t.forecast_month
                    ORDER BY hash(cast(t.cdi_id as string)), t.cdi_id
                ) AS rn
            FROM {DATASET_WITH_FEATURES} t
        ) s
        WHERE s.rn <= {SAMPLE_SIZE}
    )
    SELECT 
        {cols_bin_sql}
    FROM {DATASET_WITH_FEATURES} t
    JOIN sample_keys k
      ON t.cdi_id = k.cdi_id
     AND t.forecast_month = k.forecast_month;
"""
#{cols_bin_sql}
with conn:
    conn.execute(f"drop table if exists {DATASET_BINARY_LOYAL_SAMPLE}")
    conn.execute(query)

### Full-feature copy для loyalty benchmark

Создаётся по ключам current sample, чтобы строки совпадали 1-в-1.


In [ ]:
# Full-feature candidate для dynamic FS: ТЕ ЖЕ row keys, но признаки из modular full source.
DATASET_BINARY_LOYAL_FULL = f"sandbox_crft.cb_sample_for_binary_modular_full_{postfix}"

query_full_loyal = f"""
CREATE TABLE {DATASET_BINARY_LOYAL_FULL}
STORED AS PARQUET AS
WITH sample_keys AS (
    SELECT DISTINCT cdi_id, forecast_month
    FROM {DATASET_BINARY_LOYAL_SAMPLE}
)
SELECT d.*
FROM {DATASET_WITH_FEATURES_FS} d
JOIN sample_keys k
  ON d.cdi_id = k.cdi_id
 AND d.forecast_month = k.forecast_month
"""

with conn:
    conn.execute(f"drop table if exists {DATASET_BINARY_LOYAL_FULL}")
    conn.execute(query_full_loyal)
    loyalty_row_check = conn.fetch(f"""
        SELECT 'current_common_segmentation' AS source, COUNT(*) AS rows_cnt,
               COUNT(DISTINCT cdi_id) AS clients_cnt
        FROM {DATASET_BINARY_LOYAL_SAMPLE}
        UNION ALL
        SELECT 'dynamic_modular_full' AS source, COUNT(*) AS rows_cnt,
               COUNT(DISTINCT cdi_id) AS clients_cnt
        FROM {DATASET_BINARY_LOYAL_FULL}
    """)

assert loyalty_row_check['rows_cnt'].nunique() == 1, 'Row count mismatch: current loyalty vs modular full'
assert loyalty_row_check['clients_cnt'].nunique() == 1, 'Client count mismatch: current loyalty vs modular full'
loyalty_row_check


In [ ]:
DATASET_BINARY_LOYAL_SAMPLE

## Витрина для обучения кешбэка по acrm.o_calculations

In [ ]:
# CURRENT/static feature list — оставляем как в текущем pipeline.
cols_cashback = get_feature_list('train_pipeline_cashback_columns')

cols_cashback_sql = ",".join(cols_cashback)


In [ ]:
cb_train_connected = f"sandbox_crft.cb_train_feat_select_connected_{postfix}"

drop = f"""
drop table if exists {cb_train_connected}
"""

create = f"""
create table {cb_train_connected} stored as parquet as
select
    {cols_cashback_sql}
from {DATASET_WITH_FEATURES} d
where coalesce(d.target_cashback_month, 0) >= 0 and target_loyalty_connected_flg = 1
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)

    df_check = conn.fetch(f"""
        select
            count(*) as rows_cnt,
            count(distinct cdi_id) as clients_cnt,
            count(distinct forecast_month) as months_cnt,
            avg(target_cashback_month) as avg_cashback,
            sum(target_cashback_month) as sum_cashback
        from {cb_train_connected}
    """)
# {cols_cashback_sql}
df_check

### Full-feature copy для cashback benchmark

Строки фиксируются current-таблицей, чтобы исключить изменение выборки.


In [ ]:
# Full-feature candidate для cashback dynamic FS: те же row keys, другой feature store.
cb_train_connected_full = f"sandbox_crft.cb_train_modular_full_connected_{postfix}"

create_full_cashback = f"""
CREATE TABLE {cb_train_connected_full}
STORED AS PARQUET AS
WITH current_keys AS (
    SELECT DISTINCT cdi_id, forecast_month
    FROM {cb_train_connected}
)
SELECT d.*
FROM {DATASET_WITH_FEATURES_FS} d
JOIN current_keys k
  ON d.cdi_id = k.cdi_id
 AND d.forecast_month = k.forecast_month
WHERE COALESCE(d.target_cashback_month, 0) >= 0
  AND d.target_loyalty_connected_flg = 1
"""

with conn:
    conn.execute(f"drop table if exists {cb_train_connected_full}")
    conn.execute(create_full_cashback)
    cashback_row_check = conn.fetch(f"""
        SELECT 'current_common_segmentation' AS source, COUNT(*) AS rows_cnt,
               COUNT(DISTINCT cdi_id) AS clients_cnt
        FROM {cb_train_connected}
        UNION ALL
        SELECT 'dynamic_modular_full' AS source, COUNT(*) AS rows_cnt,
               COUNT(DISTINCT cdi_id) AS clients_cnt
        FROM {cb_train_connected_full}
    """)

assert cashback_row_check['rows_cnt'].nunique() == 1, 'Row count mismatch: current cashback vs modular full'
assert cashback_row_check['clients_cnt'].nunique() == 1, 'Client count mismatch: current cashback vs modular full'
cashback_row_check


In [ ]:
# Экспорт в ./data: копируем только созданные в этом запуске витрины.
# Не удаляем предыдущие экспорты автоматически.
from pathlib import Path
import subprocess

Path('./data').mkdir(exist_ok=True)
for table in [cb_train_connected, cb_train_connected_full, DATASET_BINARY_LOYAL_SAMPLE, DATASET_BINARY_LOYAL_FULL]:
    t = table.split('.')[-1]
    src = f'/warehouse/tablespace/managed/hive/sandbox_crft.db/{t}'
    dst = Path('./data') / t
    if dst.exists():
        raise FileExistsError(f'Каталог уже существует: {dst}. Удалите его осознанно для повторного экспорта.')
    subprocess.run(['hdfs', 'dfs', '-copyToLocal', src, './data'], check=True)
